In [ ]:
import os

GROQ_KEY = "GROQ_API_KEY"

os.environ["GROQ_API_KEY"] = GROQ_KEY

print(" Groq API Key successfully locked into global environment variables!")
print(f"Current key starts with: {GROQ_KEY[:7]}...")

In [ ]:

try:
    import chromadb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "chromadb"])

try:
    import groq
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "groq"])

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rank-bm25"])

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])

try:
    from langgraph.graph import StateGraph, END
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langgraph"])

In [ ]:
import os
import json
import math
import re
from collections import deque
from typing import TypedDict, List, Dict, Any

import chromadb
from groq import Groq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 1. Setup and Initialization
# ════════════════════════════════════════════════════════════════════════════════

# [C1] MODEL constant — was never defined in v1/v2, causing immediate NameError
MODEL = "llama-3.1-8b-instant"

# [M1] Only one Groq client needed — removed unused extraction_client / question_agent_client
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "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")
client = Groq(api_key=GROQ_API_KEY)

print("Adventure begins! You can start by mentioning your name/location.")
print("(Type 'quit' or 'exit' to end the game)")
print("-" * 60)


# ════════════════════════════════════════════════════════════════════════════════
# 2. Memory Initialization
# ════════════════════════════════════════════════════════════════════════════════

# Short-term: last 10 raw messages (role/content dicts)
short_term_memory: deque = deque(maxlen=10)

# [H2] Rolling buffer: maxlen=10 because we append 2 msgs/turn (user + assistant)
#      Old value of 5 only held ~2.5 turns of context, not 5
recent_turns_buffer: deque = deque(maxlen=10)

# Long-term structured memory
long_term_memory: Dict[str, Any] = {
    "characters": {
        "player": {
            "name": None,
            "status": "healthy",
            "inventory": [],
            "location": {"name": None, "last_updated_turn": 0, "relevance_score": 1.0},
            "backstory": None,
            "last_updated_turn": 0,
            "relevance_score": 1.0,
        },
        "npcs": {},
    },
    "items": {},
    "locations": {},
    "events": [],
    "decisions": [],
    "quest_log": {
        "active": [],
        "completed": [],
        "failed": [],
    },
}

turn_count: int = 0


# ════════════════════════════════════════════════════════════════════════════════
# 3. RAG Setup — ChromaDB (Vector) + BM25 (Keyword) Hybrid
# ════════════════════════════════════════════════════════════════════════════════

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()
try:
    collection = chroma_client.get_collection("dm_memories")
except Exception:
    collection = chroma_client.create_collection("dm_memories")

_bm25_corpus: List[List[str]] = []
_bm25_docs:   List[str]       = []
_bm25_index:  BM25Okapi | None = None

# [M3] Dirty flag — BM25 only rebuilt when a query is about to fire, not on every upsert
_bm25_dirty: bool = False

# [M4] Cached ChromaDB count — avoids a round-trip on every retrieve_hybrid call
_chroma_count: int = 0


def _tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())


def _rebuild_bm25_if_needed() -> None:
    """Lazily rebuild BM25 only when dirty, not after every single upsert."""
    global _bm25_index, _bm25_dirty
    if _bm25_dirty and _bm25_corpus:
        _bm25_index = BM25Okapi(_bm25_corpus)
        _bm25_dirty = False


def update_vector_memory(memory_dict: dict, turn: int) -> None:
    """
    Upsert LTM entries into ChromaDB and BM25.
    [M3] Sets _bm25_dirty instead of rebuilding immediately.
    [M4] Updates _chroma_count cache.
    """
    global _bm25_dirty, _chroma_count

    texts, ids, metas = [], [], []

    for section, entries in memory_dict.items():
        if section == "quest_log":
            for status, quests in entries.items():
                for idx, q in enumerate(quests):
                    text = f"quest_log::{status}::{idx}::{json.dumps(q)}"
                    texts.append(text)
                    ids.append(f"quest_{status}_{idx}")
                    metas.append({"section": "quest_log", "status": status, "turn": turn})
        elif isinstance(entries, dict):
            for key, val in entries.items():
                if isinstance(val, dict):
                    for subkey, subval in val.items():
                        text = f"{section}::{key}::{subkey}::{json.dumps(subval)}"
                        texts.append(text)
                        ids.append(f"{section}_{key}_{subkey}")
                        metas.append({"section": section, "key": f"{key}.{subkey}", "turn": turn})
                else:
                    text = f"{section}::{key}::{json.dumps(val)}"
                    texts.append(text)
                    ids.append(f"{section}_{key}")
                    metas.append({"section": section, "key": key, "turn": turn})
        elif isinstance(entries, list):
            for idx, entry in enumerate(entries):
                text = f"{section}::{idx}::{json.dumps(entry)}"
                texts.append(text)
                ids.append(f"{section}_{idx}")
                metas.append({"section": section, "index": idx, "turn": turn})

    if not texts:
        return

    embeddings = embedder.encode(texts).tolist()
    collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metas)
    _chroma_count = collection.count()   # [M4] refresh cache after upsert

    existing = set(_bm25_docs)
    added_any = False
    for t in texts:
        if t not in existing:
            _bm25_corpus.append(_tokenize(t))
            _bm25_docs.append(t)
            added_any = True

    if added_any:
        _bm25_dirty = True   # [M3] mark dirty; rebuild deferred until next query


def retrieve_hybrid(query: str, top_k: int = 4) -> str:
    """Hybrid BM25 + ChromaDB retrieval via Reciprocal Rank Fusion."""
    # [M4] Use cached count instead of collection.count()
    if not _bm25_docs or _chroma_count == 0:
        return ""

    # [M3] Rebuild BM25 lazily only when needed
    _rebuild_bm25_if_needed()

    query_tokens = _tokenize(query)

    bm25_scores = _bm25_index.get_scores(query_tokens) if _bm25_index else []
    bm25_ranked = sorted(
        range(len(_bm25_docs)), key=lambda i: bm25_scores[i], reverse=True
    )[:top_k * 2]

    q_emb = embedder.encode([query]).tolist()
    vec_results = collection.query(query_embeddings=q_emb, n_results=min(top_k * 2, _chroma_count))
    vec_docs = vec_results["documents"][0] if vec_results["documents"] else []

    rrf_scores: Dict[str, float] = {}
    K = 60
    for rank, idx in enumerate(bm25_ranked):
        doc = _bm25_docs[idx]
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 0.6 / (K + rank + 1)
    for rank, doc in enumerate(vec_docs):
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 0.4 / (K + rank + 1)

    top_docs = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return "\n".join(top_docs)


# [L2] Decay rates are now configurable per section instead of a single hardcoded default
DECAY_RATES: Dict[str, float] = {
    "events":     0.08,   # events fade faster
    "decisions":  0.05,
    "npcs":       0.04,   # NPC relationships persist longer
    "items":      0.06,
    "locations":  0.03,   # locations fade the slowest
    "default":    0.05,
}


def calculate_relevance(
    last_updated_turn: int,
    current_turn: int,
    base_relevance: float,
    section: str = "default",
) -> float:
    """Exponential decay with per-section rates."""
    decay_rate = DECAY_RATES.get(section, DECAY_RATES["default"])
    age = current_turn - last_updated_turn
    if age <= 0:
        return base_relevance
    return max(0.0, base_relevance * math.exp(-decay_rate * age))


def estimate_chars(messages: list) -> int:
    return sum(len(m.get("content", "")) for m in messages)


# ════════════════════════════════════════════════════════════════════════════════
# 4. Agent State Definition
# ════════════════════════════════════════════════════════════════════════════════

class AgentState(TypedDict):
    user_input:          str
    short_term_memory:   deque
    recent_turns_buffer: deque
    long_term_memory:    dict
    turn_count:          int
    ai_response:         str
    collection:          chromadb.Collection
    prompt_char_count:   int


# ════════════════════════════════════════════════════════════════════════════════
# 5. Helper — Compact LTM Summary
# ════════════════════════════════════════════════════════════════════════════════

def build_ltm_summary(ltm: dict, current_turn: int) -> str:
    p = ltm["characters"]["player"]
    lines = [
        f"[WORLD STATE — Turn {current_turn}]",
        f"Player: {p.get('name') or 'unknown'} | "
        f"Status: {p.get('status') or 'unknown'} | "
        f"Location: {p.get('location', {}).get('name') or 'unknown'}",
        f"Inventory: {', '.join(p.get('inventory') or []) or 'empty'}",
    ]

    npcs = ltm["characters"]["npcs"]
    if npcs:
        npc_lines = []
        for name, data in npcs.items():
            # [L2] Pass section="npcs" for correct decay rate
            rel = calculate_relevance(
                data.get("last_updated_turn", 0), current_turn,
                data.get("relevance_score", 0.5), section="npcs"
            )
            if rel > 0.1:
                rel_str  = data.get("relationship", "neutral")
                pers_str = data.get("personality", "")
                npc_lines.append(f"  • {name} [{rel_str}] — {pers_str}")
        if npc_lines:
            lines.append("NPCs:\n" + "\n".join(npc_lines))

    active = ltm["quest_log"]["active"]
    if active:
        q_lines = []
        for q in active:
            remaining = [o for o in q.get("objectives", [])
                         if o not in q.get("completed_objectives", [])]
            q_lines.append(f"  • [{q['id']}] {q['title']} — Next: {remaining[0] if remaining else 'unclear'}")
        lines.append("Active Quests:\n" + "\n".join(q_lines))

    recent_events = ltm["events"][-3:]
    if recent_events:
        ev_lines = [f"  • {e.get('summary', '')[:120]}" for e in recent_events]
        lines.append("Recent Events:\n" + "\n".join(ev_lines))

    return "\n".join(lines)


# ════════════════════════════════════════════════════════════════════════════════
# 6. Node 1 — cleaner_node  (every 30 turns, 0 API calls)
# ════════════════════════════════════════════════════════════════════════════════

def cleaner_node(state: AgentState) -> AgentState:
    if state["turn_count"] % 30 != 0:
        return state

    print("[System] Pruning stale memory entries...")
    ltm   = state["long_term_memory"]
    turn  = state["turn_count"]
    thr   = 0.1

    # [L2] Each section uses its own decay rate
    ltm["items"] = {
        k: v for k, v in ltm["items"].items()
        if calculate_relevance(v.get("last_updated_turn", 0), turn,
                               v.get("relevance_score", 0.5), "items") > thr
    }
    ltm["locations"] = {
        k: v for k, v in ltm["locations"].items()
        if calculate_relevance(v.get("last_updated_turn", 0), turn,
                               v.get("relevance_score", 0.5), "locations") > thr
    }
    ltm["characters"]["npcs"] = {
        k: v for k, v in ltm["characters"]["npcs"].items()
        if calculate_relevance(v.get("last_updated_turn", 0), turn,
                               v.get("relevance_score", 0.5), "npcs") > thr
    }
    ltm["events"] = [
        e for e in ltm["events"]
        if calculate_relevance(e.get("last_updated_turn", 0), turn,
                               e.get("relevance_score", 0.5), "events") > thr
    ]
    ltm["decisions"] = [
        d for d in ltm["decisions"]
        if calculate_relevance(d.get("last_updated_turn", 0), turn,
                               d.get("relevance_score", 0.5), "decisions") > thr
    ]

    state["long_term_memory"] = ltm
    print("[System] Memory pruned.")
    return state


# ════════════════════════════════════════════════════════════════════════════════
# 7. Node 2 — dm_node  (1 API call/turn)
# ════════════════════════════════════════════════════════════════════════════════

# [L1] Approx token limit warning threshold (chars ÷ 4 ≈ tokens; llama-3.1-8b = 128k ctx)
_PROMPT_CHAR_WARN = 400_000   # ~100k tokens — warn before hitting the limit

def dm_node(state: AgentState) -> AgentState:
    user_input = state["user_input"]
    ltm        = state["long_term_memory"]
    turn       = state["turn_count"]

    ltm_block       = build_ltm_summary(ltm, turn)
    retrieved       = retrieve_hybrid(user_input, top_k=4)
    retrieval_block = f"\n[RETRIEVED CONTEXT]\n{retrieved}" if retrieved else ""

    system_content = (
        "You are a creative, immersive Dungeon Master. "
        "Keep responses rich but concise (2-4 paragraphs). "
        "Maintain strict continuity with the world state below. "
        "Reference NPCs by name and personality. "
        "Acknowledge active quests naturally when relevant.\n\n"
        f"{ltm_block}"
        f"{retrieval_block}"
    )

    # [C2] User message is appended here inside the node — NOT in the game loop
    #      This was the double-append bug: outer loop appended, then graph copy did too
    messages = [{"role": "system", "content": system_content}]
    messages += list(state["short_term_memory"])
    messages.append({"role": "user", "content": user_input})

    # [L1] Warn if approaching context limit
    prompt_char_count = estimate_chars(messages)
    if prompt_char_count > _PROMPT_CHAR_WARN:
        approx_tokens = prompt_char_count // 4
        print(f"[System] WARNING: Prompt is ~{approx_tokens:,} tokens — approaching context limit.")

    response = client.chat.completions.create(
        messages=messages,
        model=MODEL,
        max_tokens=600,
    )

    ai_response = response.choices[0].message.content
    print(f"\nDM: {ai_response}\n")

    # Update STM — append BOTH user msg and assistant reply here
    state["short_term_memory"].append({"role": "user",      "content": user_input})
    state["short_term_memory"].append({"role": "assistant", "content": ai_response})
    state["recent_turns_buffer"].append({"role": "user",      "content": user_input})
    state["recent_turns_buffer"].append({"role": "assistant", "content": ai_response})

    state["ai_response"]       = ai_response
    state["prompt_char_count"] = prompt_char_count
    return state


# ════════════════════════════════════════════════════════════════════════════════
# 8. Node 3 — unified_extractor_node  (1 API call every 5 turns)
# ════════════════════════════════════════════════════════════════════════════════

_EXTRACTOR_SYSTEM = """You are a structured memory extraction engine for a Dungeon Master AI.
Extract and return ONLY a JSON object — no preamble, no markdown fences.

Schema:
{
  "characters": {
    "player": {
      "name": string | null,
      "status": string,
      "inventory": [string],
      "location": {"name": string | null, "last_updated_turn": int, "relevance_score": float},
      "backstory": string | null,
      "last_updated_turn": int,
      "relevance_score": float
    },
    "npcs": {
      "<npc_name>": {
        "first_seen_turn": int,
        "last_updated_turn": int,
        "relevance_score": float,
        "location": string,
        "personality": string,
        "relationship": string,
        "dialogue_history": [string],
        "knows_about_player": [string]
      }
    }
  },
  "items": {
    "<item_name>": {"description": string, "location": string, "last_updated_turn": int, "relevance_score": float}
  },
  "locations": {
    "<place_name>": {"description": string, "visited": bool, "last_updated_turn": int, "relevance_score": float}
  },
  "events": [
    {"summary": string, "turn_recorded": int, "last_updated_turn": int, "relevance_score": float, "type": string}
  ],
  "decisions": [
    {"description": string, "outcome": string, "turn_recorded": int, "last_updated_turn": int, "relevance_score": float}
  ],
  "quest_log": {
    "active": [
      {"id": string, "title": string, "description": string,
       "objectives": [string], "completed_objectives": [string],
       "turn_started": int, "last_updated_turn": int}
    ],
    "completed": [{"id": string, "title": string, "turn_completed": int}],
    "failed":    [{"id": string, "title": string, "turn_failed": int}]
  },
  "summary": string
}

Rules:
- relevance_score: 1.0 = critical, 0.7 = important, 0.4 = minor, 0.1 = trivial
- Preserve existing memory; only update fields that changed
- dialogue_history: keep the 5 most recent meaningful quotes/actions per NPC
- summary: 2-3 sentence narrative summary of the recent turns
- Return valid JSON only — no markdown, no backticks, no explanation."""


def _parse_json_safe(raw: str) -> dict | None:
    """
    [M2] Two-stage JSON parser:
      1. Direct json.loads (works when response_format=json_object is supported)
      2. Regex fallback to extract {...} block (works when model ignores the format hint)
    """
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def _filter_ltm_for_extractor(ltm: dict, current_turn: int, window: int = 10) -> dict:
    """
    [H1] Returns a slimmed LTM that only includes recently-updated entries.
    Prevents the extractor prompt from ballooning with stale data over long sessions.
    Cutoff: entries updated within the last `window` turns are included.
    The player character and active quests are always included in full.
    """
    cutoff = current_turn - window

    filtered = {
        "characters": {
            "player": ltm["characters"]["player"],   # always include player
            "npcs": {
                k: v for k, v in ltm["characters"]["npcs"].items()
                if v.get("last_updated_turn", 0) >= cutoff
            },
        },
        "items": {
            k: v for k, v in ltm["items"].items()
            if v.get("last_updated_turn", 0) >= cutoff
        },
        "locations": {
            k: v for k, v in ltm["locations"].items()
            if v.get("last_updated_turn", 0) >= cutoff
        },
        "events": [
            e for e in ltm["events"]
            if e.get("last_updated_turn", 0) >= cutoff
        ],
        "decisions": [
            d for d in ltm["decisions"]
            if d.get("last_updated_turn", 0) >= cutoff
        ],
        "quest_log": ltm["quest_log"],   # always include full quest log
    }
    return filtered


def unified_extractor_node(state: AgentState) -> AgentState:
    if state["turn_count"] % 5 != 0:
        return state

    print("[System] Running unified extractor...")
    ltm    = state["long_term_memory"]
    recent = list(state["recent_turns_buffer"])
    turn   = state["turn_count"]

    # [H1] Only send recently-updated LTM to the extractor, not the full blob
    filtered_ltm = _filter_ltm_for_extractor(ltm, turn, window=10)

    user_prompt = (
        f"Current turn: {turn}\n\n"
        f"Recent Memory (last 10 turns only):\n{json.dumps(filtered_ltm, indent=2)}\n\n"
        f"Recent Turns:\n{json.dumps(recent, indent=2)}\n\n"
        "Extract and return the updated memory JSON."
    )

    try:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": _EXTRACTOR_SYSTEM},
                {"role": "user",   "content": user_prompt},
            ],
            model=MODEL,
            max_tokens=1500,
            response_format={"type": "json_object"},
        )
        raw  = response.choices[0].message.content
        # [M2] Use safe parser with regex fallback
        data = _parse_json_safe(raw)
        if data is None:
            print("[System] Extractor: could not parse JSON response — skipping update.")
            return state
    except Exception as exc:
        print(f"[System] Extractor error: {exc}")
        return state

    # --- Merge characters ---
    if "characters" in data:
        chars = data["characters"]
        if "player" in chars:
            ltm["characters"]["player"].update(chars["player"])
        if "npcs" in chars:
            for npc_name, npc_data in chars["npcs"].items():
                if npc_name in ltm["characters"]["npcs"]:
                    ltm["characters"]["npcs"][npc_name].update(npc_data)
                else:
                    ltm["characters"]["npcs"][npc_name] = npc_data

    # --- Merge flat sections ---
    for section in ["items", "locations"]:
        if section in data:
            ltm[section].update(data[section])

    # --- Append events & decisions (avoid exact duplicates) ---
    existing_event_summaries = {e.get("summary") for e in ltm["events"]}
    for event in data.get("events", []):
        if event.get("summary") not in existing_event_summaries:
            ltm["events"].append(event)

    existing_decision_descs = {d.get("description") for d in ltm["decisions"]}
    for decision in data.get("decisions", []):
        if decision.get("description") not in existing_decision_descs:
            ltm["decisions"].append(decision)

    # --- Merge quest log ---
    if "quest_log" in data:
        ql_new = data["quest_log"]

        existing_ids = {q["id"]: i for i, q in enumerate(ltm["quest_log"]["active"])}
        for q in ql_new.get("active", []):
            if q["id"] in existing_ids:
                ltm["quest_log"]["active"][existing_ids[q["id"]]].update(q)
            else:
                ltm["quest_log"]["active"].append(q)

        completed_ids = {q["id"] for q in ql_new.get("completed", [])}
        failed_ids    = {q["id"] for q in ql_new.get("failed", [])}

        ltm["quest_log"]["active"] = [
            q for q in ltm["quest_log"]["active"]
            if q["id"] not in completed_ids and q["id"] not in failed_ids
        ]

        # [L3] Deduplicate completed/failed lists by quest id before extending
        existing_completed_ids = {q["id"] for q in ltm["quest_log"]["completed"]}
        for q in ql_new.get("completed", []):
            if q["id"] not in existing_completed_ids:
                ltm["quest_log"]["completed"].append(q)
                existing_completed_ids.add(q["id"])

        existing_failed_ids = {q["id"] for q in ltm["quest_log"]["failed"]}
        for q in ql_new.get("failed", []):
            if q["id"] not in existing_failed_ids:
                ltm["quest_log"]["failed"].append(q)
                existing_failed_ids.add(q["id"])

    # --- Store narrative summary as an event ---
    if "summary" in data and data["summary"]:
        summary_text = f"[Summary T{turn-4}–T{turn}] {data['summary']}"
        # Avoid duplicate summaries for the same turn window
        if summary_text not in {e.get("summary") for e in ltm["events"]}:
            ltm["events"].append({
                "summary":           summary_text,
                "turn_recorded":      turn,
                "last_updated_turn":  turn,
                "relevance_score":    0.9,
                "type":               "summary",
            })
        print(f"[System] Summary: {data['summary'][:100]}...")

    update_vector_memory(ltm, turn)

    state["long_term_memory"]  = ltm
    state["recent_turns_buffer"].clear()
    return state


# ════════════════════════════════════════════════════════════════════════════════
# 9. LangGraph Pipeline
# ════════════════════════════════════════════════════════════════════════════════

graph = StateGraph(AgentState)

graph.add_node("cleaner",           cleaner_node)
graph.add_node("dm",                dm_node)
graph.add_node("unified_extractor", unified_extractor_node)

graph.set_entry_point("cleaner")
graph.add_edge("cleaner",           "dm")
graph.add_edge("dm",                "unified_extractor")
graph.add_edge("unified_extractor", END)

executor = graph.compile()


# ════════════════════════════════════════════════════════════════════════════════
# 10. Game Loop
# ════════════════════════════════════════════════════════════════════════════════

while True:
    turn_count += 1

    try:
        user_input = input(f"Turn {turn_count} — Your action: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nEnding the adventure. Goodbye!")
        break

    if not user_input:
        turn_count -= 1
        continue

    if user_input.lower() in {"quit", "exit"}:
        print("Ending the adventure. Goodbye!")
        break

    # [C2] Do NOT append user_input to short_term_memory here.
    #      dm_node now handles this internally to prevent double-appending.

    # [H3] Wrap invoke in try/except so Groq API errors don't kill the session
    try:
        state = executor.invoke({
            "user_input":          user_input,
            "short_term_memory":   short_term_memory,
            "recent_turns_buffer": recent_turns_buffer,
            "long_term_memory":    long_term_memory,
            "turn_count":          turn_count,
            "collection":          collection,
            "ai_response":         "",
            "prompt_char_count":   0,
        })
    except Exception as exc:
        print(f"\n[System] API error on turn {turn_count}: {exc}")
        print("[System] Your turn was not processed. Please try again.")
        turn_count -= 1
        continue

    # Persist mutable state back from the graph's copy
    short_term_memory   = state["short_term_memory"]
    recent_turns_buffer = state["recent_turns_buffer"]
    long_term_memory    = state["long_term_memory"]